# Abstention Machine Translation Scoring
Generates machine translation metric scores (e.g., BLEU) for sampled captions. We think these scores can be appropriate for this setting since we're looking at model self-similarity as a metric for whether they should abstain

## Load libraries

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../")

import os
import copy
import json
from datetime import datetime
import time
from tqdm.notebook import tqdm

import torch

from evaluation.evaluate_captions import (
    execute_bleu,
    execute_meteor,
    execute_rouge,
    execute_cider,
    execute_spice,
    execute_bertscore,
)

## Setup

In [ ]:
# setup pytorch for BERTScore
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # for multi-GPU systems, force single GPU
if torch.cuda.is_available():
    device_type = "cuda:0"  # force single, first GPU
elif torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"
print(f"Using device: {device_type}")

In [ ]:
MODELS = ["gpt-4.1", "gemini-2.5-flash", "llama-90B-4bit", "molmo-72B-4bit"]

## Load data

In [ ]:
# load data
with open(
    "./results/chi26-samples-mt-scored_945-images_2025-09-24_19-39-18.json",
    "r",
) as f:
    input_data_dict = json.load(f)
input_data_dict[0]

### Remove stopwords

In [ ]:
# remove stopwords before processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
stopwords = set(stopwords.words("english"))

In [ ]:
def process_tokens(sentence, stopwords):
    # get tokens
    tokens = word_tokenize(sentence)

    # strip punctuation from tokens
    words = [word.lower() for word in tokens if word.isalpha()]

    # remove stop words
    return [word for word in words if word not in stopwords]

In [ ]:
# count the words in the caption
for index, image in enumerate(input_data_dict):
    for model in MODELS:
        if model in image["captions"]:
            image["captions"][model]["greedy_response_no_stop"] = " ".join(
                process_tokens(image["captions"][model]["greedy_response"], stopwords)
            )

            samples_no_stop = []
            for sample in image["captions"][model]["samples"]:
                samples_no_stop.append(" ".join(process_tokens(sample, stopwords)))
                if len(sample) == 0:
                    print(f"Index {index} has empty sample response")
            image["captions"][model]["samples_no_stop"] = samples_no_stop

            # check length
            if len(image["captions"][model]["greedy_response"]) == 0:
                print(f"Index {index} has empty greedy response")

In [ ]:
### create a long list of candidates and references that can be passed to each scoring function
# FOR NOW: this is hardcoded to only work for GPT
model = "gpt-4.1"
candidates = []
references = []
indices = []  # used for merging results into
for index, image in enumerate(tqdm(input_data_dict)):
    # candidate sentences are the generated samples
    # references are a list of list of str, where each inner list is just the greedy response
    curr_candidates = image["captions"][model]["samples"]
    candidates += curr_candidates
    references += [[image["captions"][model]["greedy_response"]]] * len(curr_candidates)
    indices.append(len(curr_candidates))

candidates_no_stop = [" ".join(process_tokens(x, stopwords)) for x in candidates]
references_no_stop = [
    [" ".join(process_tokens(x, stopwords)) for x in ref] for ref in references
]

# Start Scoring

In [ ]:
output_scores = {}

In [ ]:
for order in range(1, 5):
    start_time = time.time()
    output_scores[f"bleu-{order}"] = execute_bleu(candidates, references, order)
    print(
        f"--- BLEU {order}: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
    )

In [ ]:
for order in range(1, 5):
    start_time = time.time()
    output_scores[f"bleu-{order}_no-stop"] = execute_bleu(
        candidates_no_stop, references_no_stop, order
    )
    print(
        f"--- BLEU {order} No Stop: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
    )

In [ ]:
start_time = time.time()
scores = execute_meteor(candidates, references)
output_scores["meteor"] = [{"score": float(x["meteor"])} for x in scores]
print(
    f"--- Meteor: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
scores = execute_rouge(candidates, references)
output_scores["rouge"] = [
    {
        "rouge1": float(x["rouge1"]),
        "rouge2": float(x["rouge2"]),
        "rougeL": float(x["rougeL"]),
        "rougeLsum": float(x["rougeLsum"]),
    }
    for x in scores
]
print(
    f"--- Rogue: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
_, scores = execute_cider(candidates, references)
output_scores["cider"] = [{"score": float(x)} for x in scores]
print(
    f"--- CIDEr: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
_, scores = execute_spice(candidates, references)
output_scores["spice"] = scores
print(
    f"--- SPICE: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
candidates_no_period = [x.replace(".", "") for x in candidates]
references_no_period = [[x.replace(".", "") for x in ref] for ref in references]
_, scores = execute_spice(candidates_no_period, references_no_period)
output_scores["spice-period"] = scores
print(
    f"--- SPICE No Period: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
candidates_semicolon = [x.replace(".", ";") for x in candidates]
references_semicolon = [[x.replace(".", ";") for x in ref] for ref in references]
_, scores = execute_spice(candidates_semicolon, references_semicolon)
output_scores["spice-semicolon"] = scores
print(
    f"--- SPICE Semicolon: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
output_scores["bertscore"] = execute_bertscore(
    candidates, references, device=device_type
)
print(
    f"--- BERTScore: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
output_scores["bertscore_idf"] = execute_bertscore(
    candidates, references, device=device_type, idf=True
)
print(
    f"--- BERTScore with IDF weighting: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
model_type = "microsoft/deberta-v3-large"
output_scores["bertscore_deberta_non_nli"] = execute_bertscore(
    candidates, references, device=device_type, model_type=model_type
)
print(
    f"--- BERTScore with {model_type}: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

start_time = time.time()
output_scores["bertscore_deberta_non_nli_idf"] = execute_bertscore(
    candidates, references, device=device_type, model_type=model_type, idf=True
)
print(
    f"--- BERTScore with {model_type} IDF weighting: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
model_type = "microsoft/mpnet-base"
output_scores["bertscore_general_model"] = execute_bertscore(
    candidates, references, device=device_type, model_type=model_type
)
print(
    f"--- BERTScore with {model_type}: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

start_time = time.time()
output_scores["bertscore_general_model_idf"] = execute_bertscore(
    candidates, references, device=device_type, model_type=model_type, idf=True
)
print(
    f"--- BERTScore with {model_type} IDF weighting: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

In [ ]:
start_time = time.time()
scores = execute_meteor(candidates_no_stop, references_no_stop)
output_scores["meteor_no-stop"] = [{"score": float(x["meteor"])} for x in scores]
print(
    f"--- Meteor No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)


start_time = time.time()
scores = execute_rouge(candidates_no_stop, references_no_stop)
output_scores["rouge_no-stop"] = [
    {
        "rouge1": float(x["rouge1"]),
        "rouge2": float(x["rouge2"]),
        "rougeL": float(x["rougeL"]),
        "rougeLsum": float(x["rougeLsum"]),
    }
    for x in scores
]
print(
    f"--- Rogue No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)

start_time = time.time()
_, scores = execute_cider(candidates_no_stop, references_no_stop)
output_scores["cider_no-stop"] = [{"score": float(x)} for x in scores]
print(
    f"--- CIDEr No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)

start_time = time.time()
_, scores = execute_spice(candidates_no_stop, references_no_stop)
output_scores["spice_no-stop"] = scores
print(
    f"--- SPICE No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)

start_time = time.time()
output_scores["bertscore_no-stop"] = execute_bertscore(
    candidates_no_stop, references_no_stop, device=device_type
)
print(
    f"--- BERTScore No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)

start_time = time.time()
output_scores["bertscore_idf_no-stop"] = execute_bertscore(
    candidates_no_stop, references_no_stop, device=device_type, idf=True
)
print(
    f"--- BERTScore with IDF weighting No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)


start_time = time.time()
model_type = "microsoft/deberta-v3-large"
output_scores["bertscore_deberta_non_nli_no-stop"] = execute_bertscore(
    candidates_no_stop, references_no_stop, device=device_type, model_type=model_type
)
print(
    f"--- BERTScore with {model_type} No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)

start_time = time.time()
output_scores["bertscore_deberta_non_nli_idf_no-stop"] = execute_bertscore(
    candidates_no_stop,
    references_no_stop,
    device=device_type,
    model_type=model_type,
    idf=True,
)
print(
    f"--- BERTScore with {model_type} IDF weighting No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)


start_time = time.time()
model_type = "microsoft/mpnet-base"
output_scores["bertscore_general_model_no-stop"] = execute_bertscore(
    candidates_no_stop, references_no_stop, device=device_type, model_type=model_type
)
print(
    f"--- BERTScore with {model_type} No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)

start_time = time.time()
output_scores["bertscore_general_model_idf_no-stop"] = execute_bertscore(
    candidates_no_stop,
    references_no_stop,
    device=device_type,
    model_type=model_type,
    idf=True,
)
print(
    f"--- BERTScore with {model_type} IDF weighting No Stop Words: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates_no_stop)} total samples) ---"
)

In [ ]:
# combine
output_start_ptr = 0
output_end_ptr = output_start_ptr
output_data_dict = copy.deepcopy(input_data_dict)
model = "gpt-4.1"

metrics = [
    "bleu-1",
    "bleu-2",
    "bleu-3",
    "bleu-4",
    "bleu-1_no-stop",
    "bleu-2_no-stop",
    "bleu-3_no-stop",
    "bleu-4_no-stop",
    "meteor",
    "meteor_no-stop",
    "rouge",
    "rouge_no-stop",
    "cider",
    "cider_no-stop",
    "spice",
    "spice-period",
    "spice-semicolon",
    "spice_no-stop",
    "bertscore",
    "bertscore_no-stop",
    "bertscore_idf",
    "bertscore_idf_no-stop",
    "bertscore_deberta_non_nli",
    "bertscore_deberta_non_nli_no-stop",
    "bertscore_deberta_non_nli_idf",
    "bertscore_deberta_non_nli_idf_no-stop",
    "bertscore_general_model",
    "bertscore_general_model_no-stop",
    "bertscore_general_model_idf",
    "bertscore_general_model_idf_no-stop",
]

for input_index in range(len(output_data_dict)):
    curr_index = indices[input_index]
    output_end_ptr = output_start_ptr + curr_index

    # check if metrics are in captions
    if "metrics" not in output_data_dict[input_index]["captions"][model]:
        output_data_dict[input_index]["captions"][model]["metrics"] = {}

    for metric in metrics:
        if metric in output_data_dict[input_index]:
            continue
        output_data_dict[input_index]["captions"][model]["metrics"][metric] = (
            output_scores[metric][output_start_ptr:output_end_ptr]
        )

    # increment the start pointer
    output_start_ptr = output_end_ptr

In [ ]:
# save as json
os.makedirs("results", exist_ok=True)
with open(
    f"results/chi26-samples-mt-scored_{len(output_data_dict)}-images_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.json",
    "w",
) as f:
    json.dump(output_data_dict, f, indent=2)